# Course 9: Mathematics and AI
## Session 7: Architectures from structure

**Goal:** derive weight sharing by solving 'commute with the symmetry' as a linear system, and check the structural law behind CNNs, RNNs and GNNs.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every Python cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a library, a GPU or a Lean installation. Lean code in this course was written carefully but was NOT compiled here: check it in Lean before you teach it.

### 1. One recipe
Pick the structure the data has, and demand that the layer preserves it: `layer(g · x) = g · layer(x)`. For a linear layer `W`, that is `W T = T W` for every symmetry `T`. Solve it and the architecture appears.

In [ ]:
import itertools, numpy as np
def commuting_maps(transforms, n):
    """All n x n matrices W with W T = T W for every T: the null space of a linear system."""
    M = np.vstack([np.kron(np.eye(n), T) - np.kron(T.T, np.eye(n)) for T in transforms])
    _, s, Vt = np.linalg.svd(M)
    return [v.reshape(n, n, order="F") for v in Vt[np.sum(s > 1e-9):]]

### 2. Translations give convolution
Demand that W commutes with the cyclic shift of a 1-D signal.

In [ ]:
n = 6; shift = np.roll(np.eye(n), 1, axis=0)
basis = commuting_maps([shift], n)
print("free parameters:", len(basis), "out of", n * n)
rng = np.random.default_rng(0); W = sum(rng.normal() * B for B in basis)
print(np.round(W, 2))
circulant = all(np.allclose(W[i], np.roll(W[0], i)) for i in range(n))
print("every row is the first row shifted (a circular convolution):", circulant)
assert len(basis) == n and circulant

Nobody told the solver about kernels or weight sharing. Six free numbers, each row a shifted copy of the first: that is a convolution with a kernel of length 6.

### 3. Permutations give DeepSets and graph layers
Demand that W commutes with every relabelling of n set elements.

In [ ]:
n = 4; perms = [np.eye(n)[list(p)] for p in itertools.permutations(range(n))]
basis = commuting_maps(perms, n)
W = sum(rng.normal() * B for B in basis)
print("free parameters:", len(basis)); print(np.round(W, 3))
a, b = W[0, 0] - W[0, 1], W[0, 1]
print("W = a * I + b * (all ones):", np.allclose(W, a * np.eye(n) + b * np.ones((n, n))))
assert len(basis) == 2

Two numbers: one for 'myself', one for 'everyone'. Now add a graph: only relabellings that keep the adjacency matrix A fixed are symmetries, and more layers become allowed, including `W = a I + b A`, a message-passing layer.

In [ ]:
A = np.array([[0, 1, 0, 0], [1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0]], dtype=float)            # a path of 4 nodes
autos = [P for P in perms if np.allclose(P @ A @ P.T, A)]
basis_g = commuting_maps(autos, 4)
print("symmetries of the path graph:", len(autos), "| free parameters now:", len(basis_g))
M = np.vstack([B.ravel() for B in basis_g])
in_span = lambda Wc: np.linalg.matrix_rank(np.vstack([M, Wc.ravel()])) == len(basis_g)
print("the GCN-style layer a*I + b*A is among the solutions:", in_span(0.3 * np.eye(4) + 0.7 * A))
assert in_span(0.3 * np.eye(4) + 0.7 * A)

A caution: a practical GNN shares its two numbers across all graphs, which is stricter than commuting with one graph's own symmetries. The full story treats the graph as part of the input, as in Course 7.

### 4. Sequences give recurrence
The structure of a sequence is concatenation. A function that reads a sequence into a state respects it when reading `xs + ys` equals reading `xs`, then continuing with `ys`. Functions with that property are exactly folds of a one-step cell: an RNN.

In [ ]:
Wh, Wx = rng.normal(size=(3, 3)) * 0.5, rng.normal(size=(3, 2))
cell = lambda h, x: np.tanh(Wh @ h + Wx @ x)
def run(h, xs):
    for x in xs: h = cell(h, x)
    return h
xs, ys = list(rng.normal(size=(4, 2))), list(rng.normal(size=(3, 2))); h0 = np.zeros(3)
print("run(h, xs + ys) = run(run(h, xs), ys):", np.allclose(run(h0, xs + ys), run(run(h0, xs), ys)))
print("run(h, []) = h:", np.allclose(run(h0, []), h0))
assert np.allclose(run(h0, xs + ys), run(run(h0, xs), ys))

mean_pool = lambda h, xs: np.mean(xs, axis=0) if len(xs) else h          # ignores the state: not a fold of one cell
print("mean pooling satisfies the same law:", np.allclose(mean_pool(h0, xs + ys), mean_pool(mean_pool(h0, xs), ys)))

Mean pooling fails the law as written because the mean of a concatenation needs the counts. Carry (sum, count) as the state and it becomes a fold again. Choosing the state is choosing the algebra.

### 5. The same check for three real layers
A structure-preserving layer passes its test to numerical precision. A layer that almost preserves it does not.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
conv = nn.Conv1d(1, 4, 3, padding=1, padding_mode="circular"); x = torch.randn(1, 1, 16)
e_conv = (conv(torch.roll(x, 3, -1)) - torch.roll(conv(x), 3, -1)).abs().max().item()
conv_zero = nn.Conv1d(1, 4, 3, padding=1); e_zero = (conv_zero(torch.roll(x, 3, -1)) - torch.roll(conv_zero(x), 3, -1)).abs().max().item()
stride = nn.Conv1d(1, 4, 3, padding=1, padding_mode="circular", stride=2); e_stride = (stride(torch.roll(x, 1, -1)) - torch.roll(stride(x), 1, -1)[..., :8]).abs().max().item() if False else None
print(f"circular convolution, shift by 3      : error {e_conv:.1e}")
print(f"zero-padded convolution, cyclic shift : error {e_zero:.2f}   (the border breaks the symmetry)")
pool = lambda t: t[..., ::2]                                             # keep every second position
e_pool1 = (pool(conv(torch.roll(x, 1, -1))) - torch.roll(pool(conv(x)), 1, -1)).abs().max().item()
e_pool2 = (pool(conv(torch.roll(x, 2, -1))) - torch.roll(pool(conv(x)), 1, -1)).abs().max().item()
print(f"after downsampling by 2: shift by 1 -> error {e_pool1:.2f} | shift by 2 -> error {e_pool2:.1e}   (only even shifts survive)")
assert e_conv < 1e-5 and e_zero > 1e-3 and e_pool2 < 1e-5 and e_pool1 > 1e-3

Downsampling keeps only a subgroup of the symmetry. That is why classifiers built from strided convolutions are not fully shift-invariant, a fact that surprised many people when it was measured.

### TODO
1. Solve for linear maps on 3x3 images that commute with both the horizontal and the vertical cyclic shift. How many free parameters, and what are they?
2. Solve for maps from R^n to R (a 1 x n matrix) that are invariant to all permutations. What is the only solution?
3. Attention without positional encoding: is it permutation-equivariant? Test it with `nn.MultiheadAttention`.